# 🚀 CE-EEN-B0: High-Performance Skin Disease Classification

## 🎯 Optimization Goals
- **Maximize Accuracy**: Unfreezing EfficientNetB0 for fine-tuning
- **Maximize GPU Usage**: Increased batch size & mixed precision
- **Dataset**: Massive Skin Disease Balanced Dataset (262k images, 34 classes)

## ⚙️ Key Config Changes
- **Batch Size**: 64 (High GPU Load)
- **Fine-Tuning**: EfficientNetB0 layers unfrozen
- **LR Schedule**: Cosine Decay with Warmup
- **Label Smoothing**: 0.1 (Better Generalization)
- **Loss Function**: Categorical Crossentropy (One-Hot Encoded)
- **Visualization**: Live training plots enabled

---

In [ ]:
# Install visualization tool
!pip install -q livelossplot

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.applications import EfficientNetB0
from livelossplot import PlotLossesKeras

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, balanced_accuracy_score

# ==========================================
# 🚀 STRICT GPU CONFIGURATION
# ==========================================
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError("❌ NO GPU DETECTED! Stopping execution. Please enable GPU in Settings.")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

mixed_precision.set_global_policy('mixed_float16')
print(f"✓ GPU Active: {len(gpus)} devices (Training will happen on GPU)")
print("✓ Mixed Precision: ON (Float16 Tensor Cores)")

# Config
SEED = 42
IMG_SIZE = 180
BATCH_SIZE = 64      # Increased for 100% GPU Usage
EPOCHS = 40          # Increased for better convergence
LR_START = 1e-5      # Warmup start
LR_MAX = 1e-3        # Peak LR
LABEL_SMOOTH = 0.1   # Helps generalization

DATA_DIR = Path('/kaggle/input/massive-skin-disease-balanced-dataset')
MODEL_DIR = Path('/kaggle/working/models')
MODEL_DIR.mkdir(exist_ok=True, parents=True)

tf.random.set_seed(SEED)
np.random.seed(SEED)

## 1️⃣ Data Loading & Preprocessing

In [ ]:
# Get class list
class_folders = sorted([f for f in DATA_DIR.iterdir() if f.is_dir()])
class_names = [f.name for f in class_folders]
num_classes = len(class_names)
print(f"Classes: {num_classes}")

# Collect all file paths
paths, labels = [], []
for idx, folder in enumerate(class_folders):
    imgs = list(folder.glob('*.jpg')) + list(folder.glob('*.png'))
    paths.extend([str(p) for p in imgs])
    labels.extend([idx] * len(imgs))

paths = np.array(paths)
labels = np.array(labels)
print(f"Total Images: {len(paths):,}")

# Split Data (70/15/15)
train_p, temp_p, train_l, temp_l = train_test_split(
    paths, labels, test_size=0.3, stratify=labels, random_state=SEED
)
val_p, test_p, val_l, test_l = train_test_split(
    temp_p, temp_l, test_size=0.5, stratify=temp_l, random_state=SEED
)

In [ ]:
def extract_contour_and_crop(path_bytes):
    # Decode bytes to string
    path = path_bytes.numpy().decode('utf-8')
    img = cv2.imread(path)
    if img is None:
        return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return cv2.resize(rgb, (IMG_SIZE, IMG_SIZE))
    
    cnt = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(cnt)
    margin = int(0.05 * max(w, h))
    x, y = max(0, x-margin), max(0, y-margin)
    w = min(img.shape[1]-x, w+2*margin)
    h = min(img.shape[0]-y, h+2*margin)
    
    return cv2.resize(rgb[y:y+h, x:x+w], (IMG_SIZE, IMG_SIZE))

def preprocess(path, label):
    img = tf.py_function(extract_contour_and_crop, [path], tf.uint8)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    img = tf.cast(img, tf.float32) / 255.0
    
    # ONE-HOT ENCODING for Label Smoothing
    label = tf.one_hot(label, num_classes)
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, tf.random.uniform([], 0, 4, tf.int32))
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    return tf.clip_by_value(img, 0, 1), label

def make_ds(paths, labels, aug=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if aug:
        ds = ds.shuffle(20000, seed=SEED)
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if aug:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_p, train_l, aug=True)
val_ds = make_ds(val_p, val_l)
test_ds = make_ds(test_p, test_l)

print("✓ Data Pipeline Optimized (One-Hot Encoded)")

## 2️⃣ Advanced Model Architecture (Unfrozen)

In [ ]:
def build_model(n_classes):
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    
    # EfficientNetB0 - Unfrozen for Fine-Tuning
    base = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=inp)
    base.trainable = True  # 🔓 UNFREEZE for maximum accuracy
    
    x = base.output
    
    # Custom Head
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    
    # Output (Float32)
    out = layers.Dense(n_classes, activation='softmax', dtype='float32')(x)
    
    return models.Model(inp, out, name='CE_EEN_B0_FineTuned')

model = build_model(num_classes)
model.summary()

## 3️⃣ Cosine Decay LR Schedule

In [ ]:
# Cosine Decay with Warmup
def lr_schedule(epoch):
    warmup_epochs = 3
    if epoch < warmup_epochs:
        return LR_START + (LR_MAX - LR_START) * (epoch / warmup_epochs)
    else:
        progress = (epoch - warmup_epochs) / (EPOCHS - warmup_epochs)
        return LR_MAX * 0.5 * (1 + np.cos(np.pi * progress))

lr_callback = keras.callbacks.LearningRateScheduler(lr_schedule)

# Plot LR Schedule
plt.plot([lr_schedule(e) for e in range(EPOCHS)])
plt.title('Learning Rate Schedule')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.show()

## 4️⃣ Train with Live Visualization

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adamax(learning_rate=LR_START), 
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
    metrics=['accuracy']
)

# Using .keras format to avoid mixed precision serialization issues
checkpoint_path = str(MODEL_DIR / 'best_model.keras')

callbacks = [
    ModelCheckpoint(checkpoint_path, 'val_accuracy', save_best_only=True, mode='max'),
    lr_callback,
    EarlyStopping('val_loss', patience=7, restore_best_weights=True),
    PlotLossesKeras(),  # 📈 Live Plotting
    keras.callbacks.CSVLogger(str(MODEL_DIR/'training_log.csv'))  # 💾 Log to CSV
]

print(f"Training for {EPOCHS} epochs with Batch Size {BATCH_SIZE}...")
print("Live plots will appear below:")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

## 5️⃣ Evaluation & Saving

In [ ]:
# Load best model (.keras format)
model = keras.models.load_model(str(MODEL_DIR/'best_model.keras'))

# Evaluate
loss, acc = model.evaluate(test_ds)
print(f"\n🏆 Test Accuracy: {acc:.4f}")

# Detailed Report
y_true, y_pred = [], []
for imgs, labs in test_ds:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labs.numpy(), axis=1)) # Decode one-hot
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

# Save Final (.keras format)
model.save(str(MODEL_DIR/'final_model.keras'))
np.save(str(MODEL_DIR/'classes.npy'), class_names)
print("✓ All models saved in .keras format")

## 6️⃣ 📸 Inference Demo (Path Input)
Provide a path to an image to classify it.

In [ ]:
# Load Model & Classes
loaded_model = keras.models.load_model(str(MODEL_DIR/'best_model.keras'))
loaded_classes = np.load(str(MODEL_DIR/'classes.npy'))

def predict_image_path(image_path):
    if not os.path.exists(image_path):
        print(f"❌ Error: File not found at {image_path}")
        return
        
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        print("❌ Error: Could not read image file")
        return
        
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Preprocess (Contour + Resize)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(cnt)
        margin = int(0.05 * max(w, h))
        x, y = max(0, x-margin), max(0, y-margin)
        w = min(img.shape[1]-x, w+2*margin)
        h = min(img.shape[0]-y, h+2*margin)
        img_crop = img_rgb[y:y+h, x:x+w]
    else:
        img_crop = img_rgb
        
    img_resized = cv2.resize(img_crop, (IMG_SIZE, IMG_SIZE))
    img_batch = np.expand_dims(img_resized / 255.0, axis=0)
    
    # Predict
    preds = loaded_model.predict(img_batch, verbose=0)
    idx = np.argmax(preds)
    class_name = loaded_classes[idx]
    conf = preds[0][idx]
    
    # Display
    plt.figure(figsize=(6, 6))
    plt.imshow(img_crop)
    plt.title(f"Prediction: {class_name}\nConfidence: {conf:.2%}", 
              fontsize=14, color='green', fontweight='bold')
    plt.axis('off')
    plt.show()

# Example Usage: Replace with your image path
# predict_image_path('/kaggle/input/massive-skin-disease-balanced-dataset/Acne and Rosacea Photos/Acne-Rosacea-1.jpg')
print("👇 Call the function with an image path:")
print("predict_image_path('path/to/your/image.jpg')")